In [ ]:
# Cell 1 - Download and Extract Dataset
import os, zipfile, shutil
!gdown 1y61cDyuO9Zrp1fSchWcAmCxk0B6SMx7X
with zipfile.ZipFile('/content/Project9_smart-city-traffic-patterns.zip', 'r') as z:
    z.extractall('/content/')
os.rename('/content/Project9_smart-city-traffic-patterns', '/content/smart-city-traffic-patterns')
with zipfile.ZipFile('/content/smart-city-traffic-patterns/Project9_smart-city-traffic-patterns.zip', 'r') as z:
    z.extractall('/content/smart-city-traffic-patterns/')
print('Dataset ready!')

In [ ]:
# Cell 2 - Load and Prepare Data
import pandas as pd
import numpy as np
df = pd.read_csv('/content/smart-city-traffic-patterns/smart-city-traffic-patterns/train_aWnotuB.csv')
df['DateTime'] = pd.to_datetime(df['DateTime'])
df['Hour'] = df['DateTime'].dt.hour
df['DayOfWeek'] = df['DateTime'].dt.dayofweek
df['Month'] = df['DateTime'].dt.month
df['Year'] = df['DateTime'].dt.year
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
print('Data loaded!')
print(df.shape)
print(df.head())

In [ ]:
# Cell 3 - Visualizations
import matplotlib.pyplot as plt
df.groupby('Hour')['Vehicles'].mean().plot(kind='bar', color='steelblue', figsize=(12,5))
plt.title('Average Traffic by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Average Vehicles')
plt.tight_layout()
plt.savefig('/content/traffic_by_hour.png')
plt.show()
df.groupby('Junction')['Vehicles'].mean().plot(kind='bar', color='coral', figsize=(8,5))
plt.title('Average Traffic per Junction')
plt.tight_layout()
plt.savefig('/content/traffic_by_junction.png')
plt.show()
df.groupby('IsWeekend')['Vehicles'].mean().plot(kind='bar', color=['green','orange'], figsize=(6,5))
plt.title('Weekday vs Weekend Traffic')
plt.xticks([0,1], ['Weekday','Weekend'], rotation=0)
plt.tight_layout()
plt.savefig('/content/weekday_vs_weekend.png')
plt.show()
print('All graphs saved!')

In [ ]:
# Cell 4 - Save Cleaned Data
df.to_csv('/content/traffic_cleaned.csv', index=False)
print('Cleaned data saved!')

In [ ]:
# Cell 5 - Build ML Model
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
features = ['Hour', 'DayOfWeek', 'Month', 'Year', 'IsWeekend', 'Junction']
X = df[features]
y = df['Vehicles']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Training model... please wait...')
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print('Model trained successfully!')

In [ ]:
# Cell 6 - Evaluate Model
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
print('Model Performance:')
print(f'MAE  : {mae:.2f}')
print(f'RMSE : {rmse:.2f}')

In [ ]:
# Cell 7 - Predictions Graph
plt.figure(figsize=(12,5))
plt.plot(y_test.values[:100], label='Actual', color='blue')
plt.plot(predictions[:100], label='Predicted', color='red', linestyle='--')
plt.title('Actual vs Predicted Traffic')
plt.xlabel('Sample')
plt.ylabel('Vehicles')
plt.legend()
plt.tight_layout()
plt.savefig('/content/model_predictions.png')
plt.show()
print('Graph saved!')

In [ ]:
# Cell 8 - Save Model
import pickle
with open('/content/traffic_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved!')

In [ ]:
# Cell 9 - Push to GitHub
import os
import shutil
os.environ['GITHUB_TOKEN'] = 'YOUR_TOKEN'
if os.path.exists('/content/smart-city-traffic-forecasting'):
    shutil.rmtree('/content/smart-city-traffic-forecasting')
!git clone https://$GITHUB_TOKEN@github.com/Jyothipushya/smart-city-traffic-forecasting.git
shutil.copy('/content/traffic_cleaned.csv', '/content/smart-city-traffic-forecasting/')
shutil.copy('/content/traffic_by_hour.png', '/content/smart-city-traffic-forecasting/')
shutil.copy('/content/traffic_by_junction.png', '/content/smart-city-traffic-forecasting/')
shutil.copy('/content/weekday_vs_weekend.png', '/content/smart-city-traffic-forecasting/')
shutil.copy('/content/model_predictions.png', '/content/smart-city-traffic-forecasting/')
shutil.copy('/content/traffic_model.pkl', '/content/smart-city-traffic-forecasting/')
shutil.copy('/content/traffic_forecasting_clean.ipynb', '/content/smart-city-traffic-forecasting/traffic_forecasting.ipynb')
print('All files copied!')

In [ ]:
# Cell 10 - Commit and Push
%cd /content/smart-city-traffic-forecasting
!git config --global user.email 'jyothipushya@gmail.com'
!git config --global user.name 'Jyothipushya'
!git add .
!git commit -m 'Week 1 and Week 2 - Complete Project Code'
!git push https://$GITHUB_TOKEN@github.com/Jyothipushya/smart-city-traffic-forecasting.git main